### ספריות


In [1]:
import pandas as pd
import os
import sys

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

### העלת משתנים להרצת הקוד


In [2]:
# מיקום תיקייה נוכחית
cwd = os.getcwd()

education_folder_path = os.path.dirname(cwd)

In [3]:
# תאריך
file_date=pd.Timestamp.today().strftime('%y%m%d')

### פונקציות גלובליות


In [4]:
# הוספת נתיב modules כנתיב יחסי
sys.path.append('../modules')

from global_functions import remove_spaces_in_columns, up_load_df

### העלאת טבלאות


In [5]:
# בתי ספר ממשרד החינוך בכל הארץ
Gschool=up_load_df(r'{}\background_files\ministry_of_education\מוסדות'.format(education_folder_path),'schools_2023')
Gschool = remove_spaces_in_columns(Gschool)
Gschool['מספר_תלמידים'] = Gschool['מספר_תלמידים'].fillna(0)

In [6]:
# יישובים בשטח צתאל שם וקוד יישוב
JTMT_setls=up_load_df(r'{}\background_files'.format(education_folder_path),'210615_מקוצר_רשימת_יישובים_באחריות_צתאל')
JTMT_setls = remove_spaces_in_columns(JTMT_setls)

In [7]:
# קאורדינטות של מוסדות חינוך במרחב ירושלים
moe_mosdot_coordinates_2022=up_load_df(r'{}\background_files\ministry_of_education\מוסדות'.format(education_folder_path),'moe_mosdot_coordinates_2022')
moe_mosdot_coordinates_2022=remove_spaces_in_columns(moe_mosdot_coordinates_2022)

### עיבוד


In [8]:
# סינון ליישובים שנמצאים בשטח צתאל
Gschool=Gschool[Gschool['יישוב'].isin(JTMT_setls['שם_יישוב'])]

In [9]:
# סינון של מוסדות בירושלים ובית שמש
JLM=Gschool[Gschool['יישוב'] == 'ירושלים']
BShemesh=Gschool[Gschool['יישוב'] == 'בית שמש']
Gschool=Gschool[Gschool['יישוב'] != 'ירושלים']
Gschool=Gschool[Gschool['יישוב'] != 'בית שמש']

In [10]:
moe_mosdot_coordinates_2022 = moe_mosdot_coordinates_2022.rename(columns={'SEMEL_MOSAD': 'סמל_מוסד'})
moe_mosdot_coordinates_2022 = moe_mosdot_coordinates_2022.rename(columns={'ITM_X': 'coordinate_x'})
moe_mosdot_coordinates_2022 = moe_mosdot_coordinates_2022.rename(columns={'ITM_Y': 'coordinate_y'})

In [11]:
# מיזוג טבלאות המוסדות והקאורדינטות לפי העמודה "סמל_מוסד"
Gschool = pd.merge(Gschool, moe_mosdot_coordinates_2022[['סמל_מוסד', 'coordinate_x', 'coordinate_y']],
                     on='סמל_מוסד', how='left')

BShemesh = pd.merge(BShemesh, moe_mosdot_coordinates_2022[['סמל_מוסד', 'coordinate_x', 'coordinate_y']],
                     on='סמל_מוסד', how='left')

JLM = pd.merge(JLM, moe_mosdot_coordinates_2022[['סמל_מוסד', 'coordinate_x', 'coordinate_y']],
                     on='סמל_מוסד', how='left')

In [12]:
Gschool=Gschool.drop(columns=['כתובת_חט"ב', "טלפון", 'דוא"ל_מזכירות','סוג_חינוך', "מעמד_משפטי", "סוג_פיקוח", "מגזר", "יחידת_דיווח", "שכבה", "מספר_תלמידים"])
BShemesh=BShemesh.drop(columns=['כתובת_חט"ב', "טלפון", 'דוא"ל_מזכירות','סוג_חינוך', "מעמד_משפטי", "סוג_פיקוח", "מגזר", "יחידת_דיווח", "שכבה", "מספר_תלמידים"])
JLM=JLM.drop(columns=['כתובת_חט"ב', "טלפון", 'דוא"ל_מזכירות','סוג_חינוך', "מעמד_משפטי", "סוג_פיקוח", "מגזר", "יחידת_דיווח", "שכבה", "מספר_תלמידים"])

In [13]:
GschoolNaN = Gschool[Gschool['coordinate_x'].isna() | Gschool['coordinate_y'].isna()]
Gschool = Gschool[~(Gschool['coordinate_x'].isna() | Gschool['coordinate_y'].isna())]
Gschool['SRC'] = 'moe_mosdot_coordinates'
# GschoolNaN.to_excel(r'{}\Intermediates\{}_GschoolNaN.xlsx'.format(cwd, file_date), index=False)

BShemeshNaN = BShemesh[BShemesh['coordinate_x'].isna() | BShemesh['coordinate_y'].isna()]
BShemesh = BShemesh[~(BShemesh['coordinate_x'].isna() | BShemesh['coordinate_y'].isna())]
BShemesh['SRC'] = 'moe_mosdot_coordinates'
# BShemeshNaN.to_excel(r'{}\Intermediates\{}_BShemeshNaN.xlsx'.format(cwd, file_date), index=False)

JLMNaN = JLM[JLM['coordinate_x'].isna() | JLM['coordinate_y'].isna()]
JLM = JLM[~(JLM['coordinate_x'].isna() | JLM['coordinate_y'].isna())]
JLM['SRC'] = 'moe_mosdot_coordinates'
# JLMNaN.to_excel(r'{}\Intermediates\{}_JLMNaN.xlsx'.format(cwd, file_date), index=False)

### ייצוא


In [14]:
Gschool.to_excel(r'{}\Intermediates\{}_Gschool_moe_mosdot_coordinates_2022.xlsx'.format(cwd, file_date), index=False)
BShemesh.to_excel(r'{}\Intermediates\{}_BShemesh_moe_mosdot_coordinates_2022.xlsx'.format(cwd, file_date), index=False)
JLM.to_excel(r'{}\Intermediates\{}_JLM_moe_mosdot_coordinates_2022.xlsx'.format(cwd, file_date), index=False)